# The baseline closed loop, step by step

This notebook answers the questions from the review meeting with one real run of the simplest loop:

> real syndrome data in -> controller (pulses to binary) -> syndrome packing -> Buffer 0 -> window manager -> weak decoder (its own memory, fetch cycles, the algorithm, release cycles, at a clock) -> committed straight to the Pauli frame.

No reorder buffer, no strong tier, no confidence estimator. Every number below is measured inside the simulator on this run; nothing is taken from a paper except the configured costs, which all live in one file, `experiments/baseline/baseline_closed_loop.yaml`.

The questions, and where each is answered:

| Question from the meeting | Section |
|---|---|
| What is fed in, what comes out | 1, 3, 10 |
| Do the operations reach the QPU through the controller | 2 |
| Where is the syndrome packing and Buffer 0 | 4 |
| Does the decoder have its own memory; is the data dropped there before compute | 7 |
| Fetch cycles, compute, release cycles at a clock, reported as time | 8 |
| Latency at every point of the path | 11 |
| Throughput and latency vs input round frequency | 12 |
| Is the decoder output correct (LER vs whole-circuit PyMatching) | 10, 13 |
| Which modules exist, which are pending | 14 |
| Open questions (confidence estimate, threshold, frontend) | 15 |


In [1]:
import sys, os
REPO = os.path.abspath(os.getcwd() if os.path.isdir("decsim") else os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, REPO); os.chdir(REPO)
from decsim.config import TICKS_PER_US
us = lambda ticks: round(ticks / TICKS_PER_US, 3)

def table(rows, columns):
    """Small aligned text table."""
    rows = [[str(r.get(c, "")) if isinstance(r, dict) else str(r[i]) for i, c in enumerate(columns)] for r in rows]
    widths = [max(len(c), *(len(r[i]) for r in rows)) if rows else len(c) for i, c in enumerate(columns)]
    line = lambda cells: "  ".join(cell.ljust(w) for cell, w in zip(cells, widths))
    print(line(columns)); print(line(["-" * w for w in widths]))
    for r in rows: print(line(r))

## 0. The configuration: every cost in one place

The yaml is the only source of numbers. Round period, decoder card, engine cycles and clock, controller times, the link cards, the Pauli frame commit.

In [2]:
print(open("experiments/baseline/baseline_closed_loop.yaml").read())

# Baseline closed loop: the single source of every parameter of the experiment.
# Every number that costs simulated time carries its source next to it.

code_task: surface_code:rotated_memory_z     # stim generator task (real detector data)
distance: 3
rounds_per_shot: 60                          # QEC rounds per shot; windows are (commit d, buffer d) sliding
noise_probability: 0.001                     # all four stim noise channels
seeds: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]        # one shot per seed at each sweep point

# Sweep axis 1: input round period (the QEC cycle time seen by the controller).
round_period_us: [1.0, 0.5, 0.2, 0.1, 0.05, 0.02]
# Sweep axis 2: weak decoder algorithm latency (one window), microseconds, or "measured":
# 0.028 = LILLIPUT [d=3, m=2] 7 cycles at 250 MHz, 2108.06569 sec. 6.3 and Table 4 (ASIC card).
# measured = wall clock of each real PyMatching call on this host (software decoder row).
algorithm_latency_us: [0.028, 0.28, measured]

controller:
  t_binary_

## 1. Tracing one shot

The real methods are wrapped so each stage records what passed through it, then one shot runs: d = 3, 9 rounds (short so every row fits on screen), p = 0.02 (noisier than the sweep so defects and corrections are visible), 1.0 us cycle, the weak decoder as real PyMatching whose algorithm stage lasts exactly the wall clock of the call.

In [3]:
from decsim.qpu.cycle_clock import QPUDevice
from decsim.controller.controller import Controller
from decsim.controller.syndrome_ingress import SyndromeIngress
from decsim.syndrome_buffer.syndrome_buffer import SyndromeBuffer
from decsim.windows.window_manager import WindowManager
from decsim.windows.window_boundaries import BoundaryCourier
from decsim.decoders.decoder_manager import DecoderManager
from decsim.decoders.decoder_memory import DecoderMemory
from decsim.pauli_frame.pauli_frame import PauliFrame

TRACE = []
CLOCK = {"engine": None}
def rec(stage, **data):
    TRACE.append(dict(t=us(CLOCK["engine"].now), stage=stage, **data))
def wrap(cls, name, before=None, after=None):
    original = getattr(cls, name)
    def wrapped(self, *args, **kwargs):
        if before: before(self, *args, **kwargs)
        result = original(self, *args, **kwargs)
        if after: after(self, result, *args, **kwargs)
        return result
    setattr(cls, name, wrapped)

def qpu_issue(self, command):
    CLOCK["engine"] = self.engine
    rec("controller -> qpu: issue", op=command.operation.name, rounds=command.round_count,
        cycle_us=us(command.round_ticks), starts_at=us(self.next_boundary()))
wrap(QPUDevice, "issue", before=qpu_issue)
def qpu_emit(self, payloads, operation):
    for p in payloads:
        rec("qpu: round emitted", op=operation.name, round=p.round_index, bits="".join(str(int(b)) for b in p.bits), size_bits=p.size_bits)
wrap(QPUDevice, "_emit", before=qpu_emit)
wrap(Controller, "accept_qpu_readout", before=lambda self, readout, route:
     rec("controller: readout accepted (pulses -> binary)", round=readout.round_index, binary_us=us(self.binary_availability_ticks)))
wrap(SyndromeIngress, "relay_qpu_readout", before=lambda self, payload, route, **kw:
     rec("packing: fragment relayed", round=payload.round_index, fragment=f"{payload.fragment_index + 1}/{payload.n_fragments}", size_bits=payload.size_bits))
def buffer_after(self, result, round_identity, **kw):
    s = self.snapshot()
    rec("buffer 0: round retained", round=round_identity[1], occupancy=s.occupancy, retained=[i[1] for i in s.retained_identities], bits="".join(map(str, result.fragments[0].bits)))
wrap(SyndromeBuffer, "finish_packing", after=buffer_after)
def wm_after(self, result, packet):
    w = [f"W{k}[{x.commit_lo}-{x.commit_hi}|buf->{x.buffer_hi}] " + ("READY" if x.t_data_complete else "waiting") for (o, k), x in sorted(self.windows.items())]
    rec("window manager: round arrived", round=packet.round_index, windows=" ".join(w))
wrap(WindowManager, "on_syndrome_arrival", after=wm_after)
wrap(DecoderManager, "enqueue", before=lambda self, job, reserve_transfer=None:
     rec("decoder manager: window enqueued", window=job.label, free_units={k: list(v) for k, v in self._free_units.items()}))
wrap(DecoderManager, "_start_job", after=lambda self, result, pool, job:
     rec("decoder manager: unit assigned", window=job.label, unit=job.unit, free_units={k: list(v) for k, v in self._free_units.items()}))
def deposit_after(self, result, job):
    rec("decoder memory: input landed in unit", unit=self.unit, window=job.label, rounds=[r.round_index for r in result.rounds], occupied_rounds=self.occupied_rounds)
wrap(DecoderMemory, "deposit", after=deposit_after)
wrap(DecoderManager, "_begin_service", before=lambda self, job: rec("decoder manager: start decode", window=job.label, unit=job.unit))
def decode_after(self, job, result):
    rec("decoder engine: result", window=job.label, syndrome_rounds_fetched=[f.round_index for f in job.payloads],
        defects=int(sum(sum(f.bits) for f in job.payloads)), correction_weight=int(sum(result.correction)) if result.correction is not None else None,
        logical=result.logical_observables, boundary_defects=result.boundary_defects)
wrap(DecoderManager, "_on_decode_done", before=decode_after)
wrap(BoundaryCourier, "send", before=lambda self, window, op, boundary, **kw:
     rec("boundary handoff (DD) at decode done", window=f"W{window.k}", to=[f"W{k}" for (_, k) in window.dependents], boundary=boundary))
wrap(PauliFrame, "commit_weak_correction", before=lambda self, **kw:
     rec("pauli frame: correction arrived (after WDO)", window=f"W{kw['window_key'][1]}", logical=kw["logical_observables"]))

from experiments.baseline.baseline_closed_loop import build_run, load_config
config = load_config("experiments/baseline/baseline_closed_loop.yaml")
config["rounds_per_shot"] = 9
config["noise_probability"] = 0.02
spec, decoder_engine = build_run(config, round_period_us=1.0, algorithm_latency_us="measured", seed=0)
done = spec.build()
print("terminal status:", done.result.terminal_status, "| events traced:", len(TRACE))

terminal status: complete | events traced: 60


## 2. Planner and controller -> QPU

The planner resolves the operation's round count and cycle and lays out the sliding windows (commit 3, buffer 3) before any data exists. The controller then commands the QPU: the operation reaches the QPU through the controller, and the QPU's cycle clock starts emitting one round per cycle.

In [4]:
op = spec.ops[0]
resolved = done.controller._resolved_operations[op.id]
print("operation:", op.name, "| qubits", op.qubits, "| patches", op.patches, "| circuit rounds", resolved.round_count, "| cycle", us(resolved.round_ticks), "us")
print("code geometry:", resolved.code_geometry)
table([dict(window=f"W{k}", commit=f"{w.commit_lo}-{w.commit_hi}", buffer_through=w.buffer_hi, rounds_read=w.n_rounds, waits_on=[f"W{d[1]}" for d in w.deps])
       for (o, k), w in sorted(done.window_manager.windows.items())], ["window", "commit", "buffer_through", "rounds_read", "waits_on"])
print()
table([r for r in TRACE if r["stage"] == "controller -> qpu: issue"], ["t", "op", "rounds", "cycle_us", "starts_at"])

operation: memory | qubits (0,) | patches (0,) | circuit rounds 9 | cycle 1.0 us
code geometry: ResolvedCodeGeometry(code_name='rotated surface code (d=3)', distance=3, commit_round_count=3, buffer_round_count=3, minimum_leading_buffer_round_count=3, minimum_trailing_buffer_round_count=3, one_patch_spatial_node_count=9, buffer_floor_override_active=False)
window  commit  buffer_through  rounds_read  waits_on
------  ------  --------------  -----------  --------
W0      1-3     6               6            []      
W1      4-9     9               6            ['W0']  

t    op      rounds  cycle_us  starts_at
---  ------  ------  --------  ---------
0.0  memory  9       1.0       0.0      


## 3. What is fed in: one syndrome round per cycle, real sampled data

Each round is the detector bits of that cycle sampled from the Stim circuit (`surface_code:rotated_memory_z`, generated at run start from the yaml parameters; the seed picks the noise realization).

In [5]:
table([r for r in TRACE if r["stage"] == "qpu: round emitted"], ["t", "op", "round", "bits", "size_bits"])

t    op      round  bits          size_bits
---  ------  -----  ------------  ---------
1.0  memory  1      0000          4        
2.0  memory  2      10110000      8        
3.0  memory  3      10100000      8        
4.0  memory  4      00000000      8        
5.0  memory  5      00000000      8        
6.0  memory  6      10000000      8        
7.0  memory  7      10000000      8        
8.0  memory  8      01001000      8        
9.0  memory  9      100100000001  12       


## 4. Controller (pulses to binary), syndrome packing, link C2B, Buffer 0

The controller accepts the readout after the QC link plus its binary-availability time (0 in this card); packing assembles the fragments of a round (one fragment per round here, so t_pack is a no-op); the packed round crosses C2B (priced: 0.10 us + bits / 1000 bits/us) into Buffer 0, which retains it until its windows are done.

In [6]:
table([r for r in TRACE if r["stage"] in ("controller: readout accepted (pulses -> binary)", "packing: fragment relayed")], ["t", "stage", "round", "binary_us", "fragment", "size_bits"])
print()
c2b = [x for x in done.result.link_traffic["transfers"] if x["path"] == "c2b"]
table([dict(round=x["attribution"]["round_lo"], sent=us(x["send_ticks"]), delivered=us(x["delivery_ticks"]), bits=x["payload_bits"], delay_us=us(x["total_delay_ticks"])) for x in c2b], ["round", "sent", "delivered", "bits", "delay_us"])
print()
table([r for r in TRACE if r["stage"] == "buffer 0: round retained"], ["t", "round", "bits", "occupancy", "retained"])

t    stage                                            round  binary_us  fragment  size_bits
---  -----------------------------------------------  -----  ---------  --------  ---------
1.0  controller: readout accepted (pulses -> binary)  1      0.0                           
1.0  packing: fragment relayed                        1                 1/1       4        
2.0  controller: readout accepted (pulses -> binary)  2      0.0                           
2.0  packing: fragment relayed                        2                 1/1       8        
3.0  controller: readout accepted (pulses -> binary)  3      0.0                           
3.0  packing: fragment relayed                        3                 1/1       8        
4.0  controller: readout accepted (pulses -> binary)  4      0.0                           
4.0  packing: fragment relayed                        4                 1/1       8        
5.0  controller: readout accepted (pulses -> binary)  5      0.0                

## 5. Window manager: when enough rounds are there

A window is ready when its commit rounds and its buffer rounds have arrived. Sliding windows are serial: W1 also waits for W0's boundary.

In [7]:
table([r for r in TRACE if r["stage"] == "window manager: round arrived"], ["t", "round", "windows"])

t      round  windows                                      
-----  -----  ---------------------------------------------
1.254  1      W0[1-3|buf->6] waiting W1[4-9|buf->9] waiting
2.258  2      W0[1-3|buf->6] waiting W1[4-9|buf->9] waiting
3.258  3      W0[1-3|buf->6] waiting W1[4-9|buf->9] waiting
4.258  4      W0[1-3|buf->6] waiting W1[4-9|buf->9] waiting
5.258  5      W0[1-3|buf->6] waiting W1[4-9|buf->9] waiting
6.258  6      W0[1-3|buf->6] READY W1[4-9|buf->9] waiting  
7.258  7      W0[1-3|buf->6] READY W1[4-9|buf->9] waiting  
8.258  8      W0[1-3|buf->6] READY W1[4-9|buf->9] waiting  
9.262  9      W0[1-3|buf->6] READY W1[4-9|buf->9] READY    


## 6. Decoder manager: the weak scheduler assigns a unit

The manager queues the ready window, finds a free weak unit, and only then asks for the transfer ("I found an ASIC, send the window").

In [8]:
table([r for r in TRACE if r["stage"] in ("decoder manager: window enqueued", "decoder manager: unit assigned")], ["t", "stage", "window", "unit", "free_units"])

t       stage                             window                  unit  free_units      
------  --------------------------------  ----------------------  ----  ----------------
6.258   decoder manager: window enqueued  memory W0 [commit 1-3]        {'default': [0]}
6.258   decoder manager: unit assigned    memory W0 [commit 1-3]  0     {'default': []} 
17.826  decoder manager: window enqueued  memory W1 [commit 4-9]        {'default': [0]}
17.826  decoder manager: unit assigned    memory W1 [commit 4-9]  0     {'default': []} 


## 7. The decoder has its own memory: transfer over CWD, then the input sits in the unit

The window travels over CWD (2.0 us here) into the assigned unit's decoder memory. Decoding cannot start before the data has landed there. This is the ASIC's memory component; its compute component is the engine in the next section.

In [9]:
cwd = [x for x in done.result.link_traffic["transfers"] if x["path"] == "cwd"]
table([dict(window=f"W{x['attribution']['window_id']}", rounds=f"{x['attribution']['round_lo']}-{x['attribution']['round_hi']}", sent=us(x["send_ticks"]), delivered=us(x["delivery_ticks"]), bits=x["payload_bits"], delay_us=us(x["total_delay_ticks"])) for x in cwd], ["window", "rounds", "sent", "delivered", "bits", "delay_us"])
print()
table([r for r in TRACE if r["stage"] == "decoder memory: input landed in unit"], ["t", "unit", "window", "rounds", "occupied_rounds"])

window  rounds  sent    delivered  bits  delay_us
------  ------  ------  ---------  ----  --------
W0      1-6     6.258   8.258      44    2.0     
W1      4-9     17.826  19.826     52    2.0     

t       unit  window                  rounds              occupied_rounds
------  ----  ----------------------  ------------------  ---------------
8.258   0     memory W0 [commit 1-3]  [1, 2, 3, 4, 5, 6]  6              
19.826  0     memory W1 [commit 4-9]  [4, 5, 6, 7, 8, 9]  6              


## 8. The decoder engine: fetch cycles, the algorithm, release cycles, at a clock

Inside the weak decoder: fetch reads the window out of the unit's memory (`fetch_cycles_per_round` cycles per round), the algorithm runs (here the real PyMatching call, its wall clock charged as simulated time; the ASIC card charges 0.028 us instead), release writes the correction out (`release_cycles_per_job`). Cycles are converted to time with `frequency_mhz` (250 MHz: one cycle = 0.004 us). The output latency is reported in time.

In [10]:
table([r for r in TRACE if r["stage"] == "decoder manager: start decode"], ["t", "window", "unit"])
print()
table([dict(window=f"W{s.window_id}", stage=s.stage, cycles=s.cycles, start=us(s.start_ticks), end=us(s.end_ticks), duration_us=us(s.end_ticks - s.start_ticks), measured_ns=s.measured_ns) for s in decoder_engine.stage_records], ["window", "stage", "cycles", "start", "end", "duration_us", "measured_ns"])
print()
table([r for r in TRACE if r["stage"] == "decoder engine: result"], ["t", "window", "syndrome_rounds_fetched", "defects", "correction_weight", "logical", "boundary_defects"])

t       window                  unit
------  ----------------------  ----
8.258   memory W0 [commit 1-3]  0   
19.826  memory W1 [commit 4-9]  0   

window  stage      cycles  start   end     duration_us  measured_ns
------  ---------  ------  ------  ------  -----------  -----------
W0      fetch      6       8.258   8.282   0.024        None       
W0      algorithm  None    8.282   17.322  9.04         9040       
W0      release    1       17.322  17.326  0.004        None       
W1      fetch      6       19.826  19.85   0.024        None       
W1      algorithm  None    19.85   27.64   7.79         7790       
W1      release    1       27.64   27.644  0.004        None       

t       window                  syndrome_rounds_fetched  defects  correction_weight  logical  boundary_defects
------  ----------------------  -----------------------  -------  -----------------  -------  ----------------
17.326  memory W0 [commit 1-3]  [1, 2, 3, 4, 5, 6]       6        3                 

## 9. Boundary to the next window (DD) and the correction to the Pauli frame (WDO)

At decode done the boundary goes to the dependent window over DD (0.5 us) and the correction goes to the Pauli frame over WDO (1.0 us); the frame commits it after `commit_us`. There is nothing between the decoder and the frame.

In [11]:
table([r for r in TRACE if r["stage"] == "boundary handoff (DD) at decode done"], ["t", "window", "to", "boundary"])
print()
dd = [x for x in done.result.link_traffic["transfers"] if x["path"] in ("dd", "wdo")]
table([dict(path=x["path"], window=f"W{x['attribution']['window_id']}", sent=us(x["send_ticks"]), delivered=us(x["delivery_ticks"]), delay_us=us(x["total_delay_ticks"])) for x in dd], ["path", "window", "sent", "delivered", "delay_us"])
print()
table([dict(window=f"W{c.window_key[1]}", accepted=us(c.accepted_ticks), committed=us(c.committed_ticks), logical=c.logical_observables) for c in done.pauli_frame.snapshot().records], ["window", "accepted", "committed", "logical"])

t       window  to      boundary                                                                                                                                            
------  ------  ------  ----------------------------------------------------------------------------------------------------------------------------------------------------
17.326  W0      ['W1']  DependencyResidual(detector_ids=(4, 6, 7, 12, 14), defects={2: [1, 0, 1, 1], 3: [1, 0, 1]})                                                         
27.644  W1      []      DependencyResidual(detector_ids=(36, 44, 53, 56, 60, 63, 71), defects={6: [1], 7: [1], 8: [0, 1, 0, 0, 1], 9: [1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1]})

path  window  sent    delivered  delay_us
----  ------  ------  ---------  --------
dd    W0      17.326  17.826     0.5     
wdo   W0      17.326  18.326     1.0     
wdo   W1      27.644  28.644     1.0     

window  accepted  committed  logical
------  --------  ---------  -------
W0      18.326    18.3

## 10. What comes out: the frame's logical prediction against the truth, and the whole timeline

In [12]:
r = done.result.operation_results[0]
print("predicted logical observables:", r.logical_observables, "| truth:", r.observable_truth, "| logical failure:", r.logical_failure)
w = sorted(done.window_manager.windows.items())
table([dict(window=f"W{k}", first_round=us(x.t_first_round), data_complete=us(x.t_data_complete), queued=us(x.t_queued), unit_assigned=us(x.t_dispatch), decode_done=us(x.t_done)) for (o, k), x in w], ["window", "first_round", "data_complete", "queued", "unit_assigned", "decode_done"])
print()
print("\n".join(done.engine.log_lines))

predicted logical observables: (1,) | truth: (1,) | logical failure: False
window  first_round  data_complete  queued  unit_assigned  decode_done
------  -----------  -------------  ------  -------------  -----------
W0      1.254        6.258          6.258   6.258          17.326     
W1      4.258        9.262          17.826  17.826         27.644     

[  0.000 us] Controller: START memory  (Clifford, qubits (0,))
[  1.000 us] QPU: memory fires round 1/9
[  1.254 us] DecoderCluster: round 1 of memory arrived (op now has rounds 1..1)
[  2.000 us] QPU: memory fires round 2/9
[  2.258 us] DecoderCluster: round 2 of memory arrived (op now has rounds 1..2)
[  3.000 us] QPU: memory fires round 3/9
[  3.258 us] DecoderCluster: round 3 of memory arrived (op now has rounds 1..3)
[  4.000 us] QPU: memory fires round 4/9
[  4.258 us] DecoderCluster: round 4 of memory arrived (op now has rounds 1..4)
[  5.000 us] QPU: memory fires round 5/9
[  5.258 us] DecoderCluster: round 5 of memory arriv

## 11. Latency at every point of the path, for this shot

The thirteen points the baseline sweep reports, measured on this run (per window: mean and max in microseconds). These are simulated times, not paper numbers.

In [13]:
from experiments.baseline.baseline_closed_loop import POINTS, collect_samples
import statistics
samples = collect_samples(done, decoder_engine)
table([dict(point=p, windows=len(v), mean_us=round(statistics.fmean(v), 3) if v else 0, max_us=round(max(v), 3) if v else 0) for p, v in samples.items()], ["point", "windows", "mean_us", "max_us"])

point                 windows  mean_us  max_us
--------------------  -------  -------  ------
c2b_per_round         9        0.108    0.112 
buffer_fill           2        5.004    5.004 
dep_block             2        4.282    8.564 
queue_wait            2        0.0      0.0   
cwd_per_window        2        2.0      2.0   
fetch                 2        0.024    0.024 
algorithm             2        8.415    9.04  
release               2        0.004    0.004 
service               2        10.443   11.068
wdo_per_window        2        1.0      1.0   
frame_commit          2        0.004    0.004 
last_round_to_frame   2        15.729   19.386
reaction_first_round  2        20.733   24.39 


## 12. Throughput and latency vs the input round frequency

A small sweep (the full one is `python -m experiments.baseline.baseline_closed_loop`): the same 60-round circuit at three round periods with the ASIC card. Faster input only grows `dep_block`, the wait for the previous window; decoded rounds per microsecond saturate at the serial chain's capacity (CWD + decode + DD per window).

In [14]:
from experiments.baseline.baseline_closed_loop import measure_shot, summarize
sweep_config = load_config("experiments/baseline/baseline_closed_loop.yaml")
rows = summarize([measure_shot(sweep_config, round_period_us=period, algorithm_latency_us=0.028, seed=seed)
                  for period in (1.0, 0.5, 0.1) for seed in (0, 1)])
table([dict(round_us=r["round_period_us"], in_rounds_per_us=round(1 / r["round_period_us"], 2), decoded_rounds_per_us=round(r["throughput_rounds_per_us"], 3),
            util=round(r["decoder_utilization"], 3), dep_block_us=round(r["dep_block_mean_us"], 2), reaction_us=round(r["reaction_first_round_mean_us"], 2), LER=r["logical_error_rate"])
       for r in rows], ["round_us", "in_rounds_per_us", "decoded_rounds_per_us", "util", "dep_block_us", "reaction_us", "LER"])

round_us  in_rounds_per_us  decoded_rounds_per_us  util   dep_block_us  reaction_us  LER
--------  ----------------  ---------------------  -----  ------------  -----------  ---
0.1       10.0              1.21                   0.788  20.3          23.86        0.0
0.5       2.0               1.163                  0.757  9.5           15.06        0.0
1.0       1.0               0.967                  0.629  0.0           8.06         0.0


## 13. Is the decoder output right

The loop's windowed decode is checked against whole-circuit PyMatching on the same shots (Gate 5 on Stim data, the Willow replay on Google's hardware data: identical shot for shot except one equal-weight tie), and the window chain timing against SWIPER (Gate 8), the links against ns-3 (Gate 6), the controller against SimPy (Gate 7). Reports: `experiments/results/validation/`.

## 14. Module status for the target architecture

| Module | Folder | State |
|---|---|---|
| Frontend (circuit and QLX programs), planner, execution runtime | `decsim/frontends/` | built; logical circuit -> Stim for a general program is the open question for the frontend |
| QPU: cycle clock, Stim / recorded / timing-only devices, code geometry | `decsim/qpu/` | built, Gates 4, 5, Willow |
| Controller: pulses to binary, syndrome packing (ingress), feedback streams | `decsim/controller/` | built, Gate 7 |
| Buffer 0 | `decsim/syndrome_buffer/` | built (unbounded in the baseline; finite slots and holds exist) |
| Window manager: sliding / parallel windows, boundaries, ledger | `decsim/windows/` | built, Gates 1, 2, 8 |
| Decoder manager: weak scheduler, units, decoder memory, engine (fetch / algorithm / release) | `decsim/decoders/` | built; PyMatching, Fusion Blossom checked (Gate 3); the ASIC card is one latency number, no internal pipeline model yet |
| Links (ten cards, serialization and queueing) | `decsim/links/` | built, Gate 6 |
| Pauli frame, conditional release | `decsim/pauli_frame/` | built (minimal frame) |
| Strong tier, weak/strong switching, escalation | `decsim/decoders/strong_escalation, weak_strong_switching` | built but off in the baseline |
| Confidence estimator, G threshold | `decsim/confidence/` | placeholder; policy not decided |
| Reorder buffer | none | not built, not in the baseline by decision |


## 15. Open questions, as agreed

- Confidence estimate and threshold: start offline, the rule from the decoder-switching paper, then show with numbers whether a fixed threshold is good enough before anything adaptive. Hardware (inside the ASIC or a separate comparator) comes after the rule is fixed.
- The weak decoder's internal pipeline: today fetch cycles + algorithm + release cycles at a clock; a fetch / decode / execute / write-back model can replace the algorithm number later without changing the rest.
- Frontend: how a logical circuit becomes a Stim circuit for a general program (QLX lowers to analytical estimates only); to discuss with Margaret.
